In [2]:
# Cell 1: Load libraries and raw data
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/2023_races_combined.csv")

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Data loaded: 200 rows, 10 columns


,DriverNumber,Abbreviation,FullName,TeamName,GridPosition,Position,Points,Status,Year,Race
0,1,VER,Max Verstappen,Red Bull Racing,1.0,1.0,25.0,Finished,2023,Bahrain
1,11,PER,Sergio Perez,Red Bull Racing,2.0,2.0,18.0,Finished,2023,Bahrain
2,14,ALO,Fernando Alonso,Aston Martin,5.0,3.0,15.0,Finished,2023,Bahrain
3,55,SAI,Carlos Sainz,Ferrari,4.0,4.0,12.0,Finished,2023,Bahrain
4,44,HAM,Lewis Hamilton,Mercedes,7.0,5.0,10.0,Finished,2023,Bahrain


In [3]:
# Cell 2: Clean and prepare data
# GridPosition 0 means they started from pit lane - replace with last position
df['GridPosition'] = df['GridPosition'].replace(0, 20)

# Convert Position to numeric (some might be NaN if they retired)
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# Create a binary column - did the driver finish in the Top 5?
df['Top5Finish'] = (df['Position'] <= 5).astype(int)

# Create a binary column - did the driver finish in the Top 10?
df['Top10Finish'] = (df['Position'] <= 10).astype(int)

# Did the driver finish the race or retire?
df['Finished'] = (df['Status'] == 'Finished').astype(int)

print("Columns after cleaning:")
print(df.columns.tolist())
print(f"\nTop 5 finishes in dataset: {df['Top5Finish'].sum()}")
print(f"Retirements in dataset: {(df['Finished'] == 0).sum()}")
df.head(10)

Columns after cleaning:
['DriverNumber', 'Abbreviation', 'FullName', 'TeamName', 'GridPosition', 'Position', 'Points', 'Status', 'Year', 'Race', 'Top5Finish', 'Top10Finish', 'Finished']

Top 5 finishes in dataset: 50
Retirements in dataset: 66


,DriverNumber,Abbreviation,FullName,TeamName,GridPosition,Position,Points,Status,Year,Race,Top5Finish,Top10Finish,Finished
0,1,VER,Max Verstappen,Red Bull Racing,1.0,1.0,25.0,Finished,2023,Bahrain,1,1,1
1,11,PER,Sergio Perez,Red Bull Racing,2.0,2.0,18.0,Finished,2023,Bahrain,1,1,1
2,14,ALO,Fernando Alonso,Aston Martin,5.0,3.0,15.0,Finished,2023,Bahrain,1,1,1
3,55,SAI,Carlos Sainz,Ferrari,4.0,4.0,12.0,Finished,2023,Bahrain,1,1,1
4,44,HAM,Lewis Hamilton,Mercedes,7.0,5.0,10.0,Finished,2023,Bahrain,1,1,1
5,18,STR,Lance Stroll,Aston Martin,8.0,6.0,8.0,Finished,2023,Bahrain,0,1,1
6,63,RUS,George Russell,Mercedes,6.0,7.0,6.0,Finished,2023,Bahrain,0,1,1
7,77,BOT,Valtteri Bottas,Alfa Romeo,12.0,8.0,4.0,Finished,2023,Bahrain,0,1,1
8,10,GAS,Pierre Gasly,Alpine,20.0,9.0,2.0,Finished,2023,Bahrain,0,1,1
9,23,ALB,Alexander Albon,Williams,15.0,10.0,1.0,Finished,2023,Bahrain,0,1,1


In [4]:
# Cell 3: Add track type feature
# Street circuits are very different from permanent tracks
# Overtaking is harder on street circuits

street_circuits = ['Monaco', 'Azerbaijan', 'Saudi Arabia', 'Miami', 'Singapore']

df['IsStreetCircuit'] = df['Race'].isin(street_circuits).astype(int)

print("Street circuits in our data:")
print(df[df['IsStreetCircuit'] == 1]['Race'].unique())
print("\nPermanent circuits in our data:")
print(df[df['IsStreetCircuit'] == 0]['Race'].unique())

Street circuits in our data:
['Saudi Arabia' 'Azerbaijan' 'Miami' 'Monaco']

Permanent circuits in our data:
['Bahrain' 'Australia' 'Spain' 'Canada' 'Britain' 'Hungary']


In [5]:
# Cell 4: Create grid position bands
# Instead of exact position, group into bands
# Front row, top 5, top 10, midfield, back

def grid_band(pos):
    if pos <= 3:
        return 'Front'      # Pole/front row
    elif pos <= 6:
        return 'TopSix'     # Strong qualifying
    elif pos <= 10:
        return 'TopTen'     # Points positions
    elif pos <= 15:
        return 'Midfield'   # Midfield
    else:
        return 'Back'       # Back of the grid

df['GridBand'] = df['GridPosition'].apply(grid_band)

print("Grid band distribution:")
print(df['GridBand'].value_counts())

Grid band distribution:
GridBand
Back        50
Midfield    50
TopTen      40
Front       30
TopSix      30
Name: count, dtype: int64


In [6]:
# Cell 5: Calculate each driver's average points per race
# This captures overall season form

driver_avg_points = df.groupby('Abbreviation')['Points'].mean().reset_index()
driver_avg_points.columns = ['Abbreviation', 'DriverAvgPoints']

df = df.merge(driver_avg_points, on='Abbreviation', how='left')

print("Top 10 drivers by average points per race:")
driver_avg_points.sort_values('DriverAvgPoints', ascending=False).head(10)

Top 10 drivers by average points per race:


,Abbreviation,DriverAvgPoints
19,VER,24.1
11,PER,14.1
5,HAM,12.7
1,ALO,12.2
14,RUS,7.8
15,SAI,6.9
7,LEC,5.5
9,NOR,4.8
17,STR,3.7
10,OCO,2.9


In [7]:
# Cell 6: Calculate team average points per race
# Team strength is a huge factor in F1

team_avg_points = df.groupby('TeamName')['Points'].mean().reset_index()
team_avg_points.columns = ['TeamName', 'TeamAvgPoints']

df = df.merge(team_avg_points, on='TeamName', how='left')

print("Teams ranked by average points per race:")
team_avg_points.sort_values('TeamAvgPoints', ascending=False)

Teams ranked by average points per race:


,TeamName,TeamAvgPoints
8,Red Bull Racing,19.10
7,Mercedes,10.25
3,Aston Martin,7.95
4,Ferrari,6.20
6,McLaren,3.75
2,Alpine,2.20
9,Williams,0.55
0,Alfa Romeo,0.45
5,Haas F1 Team,0.40
1,AlphaTauri,0.10


In [8]:
# Cell 7: Save processed data
output_path = "../data/processed/2023_features.csv"
df.to_csv(output_path, index=False)

print(f"Feature dataset saved!")
print(f"Shape: {df.shape}")
print(f"\nAll features created:")
for col in df.columns:
    print(f"  - {col}")

Feature dataset saved!
Shape: (200, 17)

All features created:
  - DriverNumber
  - Abbreviation
  - FullName
  - TeamName
  - GridPosition
  - Position
  - Points
  - Status
  - Year
  - Race
  - Top5Finish
  - Top10Finish
  - Finished
  - IsStreetCircuit
  - GridBand
  - DriverAvgPoints
  - TeamAvgPoints
